In [1]:
import os
import glob
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdMolDescriptors
import gc

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


In [2]:
INPUT_FOLDER = "datasets/*.csv"

In [3]:
OUTPUT_FOLDER = "master_toxocity_dataset.csv"

CNONICALIZE SMILES

In [4]:
def canonicalize_smiles(smiles):
    """converts SMILES with many possible forms into one standard, consistent format"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None

CLEAN SMILE -> REMOVE SALT / WATER -> AVOIDE CHOOSING WRONG FRAGMENT -> KEEP BIOLOGICAL MEANINGFULL STRUCTURE {this will imporive generalization}

In [5]:
def is_inorganic(smiles):
    """ TRUE if molecule has NO carbon atoms, FALSE otherwise """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return not any(atom.GetAtomicNum() == 6 for atom in mol.GetAtoms())

# print("Score:", is_inorganic("c1ccccc1CC(=O)O")) 


In [6]:
def fragment_score(smiles):
    """Scores the fragment based on drug likeness, higher score means more druglike"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

   
    heavy_atom = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    arom = rdMolDescriptors.CalcNumAromaticRings(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)

   
    oxygen = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 8)
    nitrogen = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)

    
    return (
        heavy_atom +
        (1.5 * arom) +
        (2.0 * rings) +
        (0.5 * oxygen) +
        (0.8 * nitrogen)
    )


#  print("Score:", fragment_score("c1ccccc1CC(=O)O")) 


In [7]:
from rdkit import Chem

def best_fragment(smiles):
    """returns the best fragment of a molecule based on drug likeness score"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = Chem.GetMolFrags(mol, asMols=True)
    organic_fragments = [f for f in fragments if not is_inorganic(Chem.MolToSmiles(f))]
    if not organic_fragments:
        organic_fragments = fragments

    
    best_frag_mol = max(organic_fragments, key=lambda f: fragment_score(Chem.MolToSmiles(f)))
    
    return Chem.MolToSmiles(best_frag_mol)





In [8]:
print(best_fragment(("[N+](=O)([O-])[O-].[Ag+]")))

O=[N+]([O-])[O-]


GENERATE MURCKO SCAFFOLD

In [9]:
def generate_scaffold(smiles):
    """
    Generates the Bemis-Murcko scaffold for a given SMILES string. 
    Returns the scaffold as a canonical SMILES string, 
    or 'NO_SCAFFOLD' if the molecule is acyclic.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        
      
        if mol is None:
            return None 
            
       
        scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
        
       
        scaffold_smiles = Chem.MolToSmiles(scaffold_mol, canonical=True)
        
       
        if scaffold_smiles == "":
            return "NO_SCAFFOLD"
            
        return scaffold_smiles
        
    except Exception as e:
        
        return None

In [10]:
print(generate_scaffold("*C(=O)[C@H](CCCCNC(=O)OCCOC)NC(=O)OCCOC"))

NO_SCAFFOLD


In [11]:
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_tautomer(smiles):
    """Forces the molecule into its canonical tautomer state to prevent duplicates."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        # Initialize the Tautomer Enumerator
        enumerator = rdMolStandardize.TautomerEnumerator()
        
        # Find the most stable, canonical tautomer
        canonical_tautomer = enumerator.Canonicalize(mol)
        
        # Return the SMILES (keeping 3D stereochemistry intact!)
        return Chem.MolToSmiles(canonical_tautomer, isomericSmiles=True)
    except:
        return None

In [12]:
def clean_chunk(df, source_name):
    """Applies  exact cleaning steps to a small chunk of data to save RAM."""
    print(f"Cleaning {source_name}...")
    
    # Canonicalize
    df['SMILES'] = df['SMILES'].apply(canonicalize_smiles)
    df = df.dropna(subset=['SMILES'])
    
    #  Best Fragment
    df['SMILES'] = df['SMILES'].apply(best_fragment)
    df = df.dropna(subset=['SMILES'])
    
    #  Re-canonicalize
    df['SMILES'] = df['SMILES'].apply(canonicalize_smiles)
    df = df.dropna(subset=['SMILES'])
    
    # Drop duplicates 
    df = df.drop_duplicates(subset=['SMILES'])

    return df

In [13]:
csv_files = glob.glob(INPUT_FOLDER)

In [14]:
print(f"Found {len(csv_files)} files to process.")

Found 18 files to process.


In [15]:
master_df = None

for file in csv_files:

    
    print(f"Processing {file}...")
    df = pd.read_csv(file)
    
   
    if 'SMILES' not in df.columns:
        print(f"Warning: 'SMILES' column not found in {file}. Skipping.")
        continue

    
    df['SMILES'] = df['SMILES'].apply(canonicalize_smiles)
    df = df.dropna(subset=['SMILES'])
    df['SMILES'] = df['SMILES'].apply(best_fragment)
    df = df.dropna(subset=['SMILES'])
    df['SMILES'] = df['SMILES'].apply(canonicalize_smiles)
   
   
    before_invalid = len(df)
    df = df.dropna(subset=['SMILES'])
    
    after_invalid = len(df)
    print(f"Invalid SMILES removed from {file}: {before_invalid - after_invalid}")

    
    before_duplicates = len(df)
    df = df.drop_duplicates(subset=['SMILES'])
    after_duplicates = len(df)
    print(f"Duplicate SMILES removed from {file}: {before_duplicates - after_duplicates}")

   
    if master_df is None:
        master_df = df
    else:
        master_df = pd.merge(master_df, df, on='SMILES', how='outer')

    del df
    gc.collect()

print(f" dataset merged successfully! Shape: {master_df.shape}")


Processing datasets/CYP450_CYP2C19.csv...
Invalid SMILES removed from datasets/CYP450_CYP2C19.csv: 0
Duplicate SMILES removed from datasets/CYP450_CYP2C19.csv: 1
Processing datasets/final_eye_irritation_master.csv...
Invalid SMILES removed from datasets/final_eye_irritation_master.csv: 0
Duplicate SMILES removed from datasets/final_eye_irritation_master.csv: 25
Processing datasets/FDA_APPROVD_clintox_df.csv...
Invalid SMILES removed from datasets/FDA_APPROVD_clintox_df.csv: 0
Duplicate SMILES removed from datasets/FDA_APPROVD_clintox_df.csv: 27
Processing datasets/Developmental and Reproductive Toxicity_Developmental Toxicity.csv...
Invalid SMILES removed from datasets/Developmental and Reproductive Toxicity_Developmental Toxicity.csv: 0
Duplicate SMILES removed from datasets/Developmental and Reproductive Toxicity_Developmental Toxicity.csv: 2
Processing datasets/SIDER.csv...
Invalid SMILES removed from datasets/SIDER.csv: 0
Duplicate SMILES removed from datasets/SIDER.csv: 36
Process

In [16]:
#  FINAL MERGE 

# FINAL MERGE: Combine all dataframes (if you have multiple, put them in the list)
final_dataset = pd.concat([master_df], ignore_index=True)

# The Ultimate Deduplication: Guarantee that the highest hazard (1.0) overrides a Safe (0.0)
final_dataset = final_dataset.groupby('SMILES', as_index=False).max()



In [17]:
final_dataset

,SMILES,CYP450_CYP2C19,Eye Irritation,CT_TOX,Developmental Toxicity,Hepatobiliary disorders,Metabolism and nutrition disorders,Product issues,Eye disorders,Investigations,...,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,CYP450_CYP2C9,Respiratory_toxicity,Cardiotoxicity
0,*C(=O)[C@H](CCCCNC(=O)OCCOC)NC(=O)OCCOC,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,*C1(*)C(*)(*)C(*)(*)C(*)(*)C(*)(*)C(*)(*)C(*)(...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,*CC(*)(C)C(=O)OCC(C)O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,*CC(*)N1CCCC1=O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,*CC(*)O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37979,c1nnc(NCNc2nncs2)s1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
37980,c1nnn[nH]1,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37981,c1scc2c1-c1cscc1C1NC21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37982,c1scc2c1-c1cscc1C1OC21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
final_dataset['scaffold'] = final_dataset['SMILES'].apply(generate_scaffold)

In [19]:
final_dataset.to_csv(OUTPUT_FOLDER, index=False)

In [20]:
final_dataset

,SMILES,CYP450_CYP2C19,Eye Irritation,CT_TOX,Developmental Toxicity,Hepatobiliary disorders,Metabolism and nutrition disorders,Product issues,Eye disorders,Investigations,...,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,CYP450_CYP2C9,Respiratory_toxicity,Cardiotoxicity,scaffold
0,*C(=O)[C@H](CCCCNC(=O)OCCOC)NC(=O)OCCOC,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_SCAFFOLD
1,*C1(*)C(*)(*)C(*)(*)C(*)(*)C(*)(*)C(*)(*)C(*)(...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C1CCCCCCCCCCC1
2,*CC(*)(C)C(=O)OCC(C)O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_SCAFFOLD
3,*CC(*)N1CCCC1=O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O=C1CCCN1
4,*CC(*)O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_SCAFFOLD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37979,c1nnc(NCNc2nncs2)s1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,c1nnc(NCNc2nncs2)s1
37980,c1nnn[nH]1,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,c1nnn[nH]1
37981,c1scc2c1-c1cscc1C1NC21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,c1scc2c1-c1cscc1C1NC21
37982,c1scc2c1-c1cscc1C1OC21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,c1scc2c1-c1cscc1C1OC21
